

# **Quantum Classifier - ansatz comparison**


**Authors:** Koło Naukowe Axion

**Dataset:** Banknote Authentication (UCI ML Repository)

**Framework:** Qiskit Machine Learning + PyTorch

**Abstract**

This notebook presents a comparative analysis of two different quantum variational architectures (ansatze) designed for a hybrid quantum-classical machine learning task. The primary focus is to evaluate performance differences between a standard simulator-optimized ansatz and a hardware-efficient ansatz specifically tailored for the IQM Spark (Odra) quantum computer.

## 1. Environment Setup
This section handles dependency installation and imports. For reproducibility, all package versions should be pinned in a production environment.




### 1.1 Package Installation (Optional)

Set `INSTALL_DEPS = True` if running in a fresh environment. For production use, pin specific versions.

In [ ]:
# @title
# Optional: Install dependencies if not already present
INSTALL_DEPS = True

if INSTALL_DEPS:
    import sys
    import subprocess

    packages = [
        'numpy',
        'scikit-learn',
        'ucimlrepo',
        'qiskit',
        'qiskit_algorithms',
        'qiskit-machine-learning',
        'qiskit-aer',
        'qiskit-ibm-runtime',
        'iqm-client[qiskit]',
        'torch',
        'matplotlib'
    ]

    for pkg in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

### 1.2 Imports


In [ ]:
# Standard library
import random

# Scientific computing
import numpy as np
import matplotlib.pyplot as plt

# Machine learning
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient

# Third-party: Quantum Hardware
from iqm.qiskit_iqm import IQMProvider


## 2. Reproducibility and Random Seed Control

In [ ]:
def set_random_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across numpy, PyTorch, and Python's random module.

    Parameters
    ----------
    seed : int
        Random seed value (default: 42)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Set global seed
RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)

## 3. Data Preparation

The **Banknote Authentication Dataset** contains features extracted from images of genuine and forged banknotes. We perform feature engineering (interaction term) and scale features to the range [0, π] for angle encoding.

In [ ]:
def prepare_data(test_size: float = 0.2, random_state: int = 42):
    # Fetch dataset from UCI repository
    banknote_authentication = fetch_ucirepo(id=267)
    X = banknote_authentication.data.features.to_numpy()
    y = banknote_authentication.data.targets.to_numpy().ravel()

    indices = np.arange(len(X))

    # Sanity checks
    assert X.shape[1] == 4, f"Expected 4 features, got {X.shape[1]}"
    assert set(np.unique(y)) == {0, 1}, f"Expected binary labels {{0, 1}}, got {set(np.unique(y))}"

    # Feature engineering: interaction term
    variance = X[:, 0].reshape(-1, 1)
    skewness = X[:, 1].reshape(-1, 1)
    interaction = variance * skewness
    X_expanded = np.hstack((X, interaction))

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_expanded, y, indices, test_size=test_size, random_state=random_state
    )

    # Scale features to [π/4, π/4] for angle encoding
    scaler = MinMaxScaler(feature_range=(-np.pi/4, np.pi/4))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Dataset loaded: {len(X_train)} train, {len(X_test)} test samples")
    print(f"Feature dimension: {X_train_scaled.shape[1]}")

    return X_train_scaled, X_test_scaled, y_train, y_test, idx_train, idx_test

# 4. Ansatz 1: Ring Topology (Simulation-Optimized)


**Overview**

This ansatz is designed to maximize expressibility and entanglement capability by leveraging a global connectivity pattern. It follows a "Circuit-centric" design philosophy, prioritizing high-dimensional state representation to achieve superior classification performance in ideal environments.

**Technical Architecture**

 * **Layered Structure:** The circuit employs a dual-layer strategy consisting of
independent rotation sub-layers ($RY$, $RX$) and sophisticated entanglement blocks.
 * **Ring Topology:** Entanglement is implemented via a circular chain where each qubit $i$ is coupled with qubit $(i+1) \pmod n$. This ensures that information from any qubit can reach any other qubit in the shortest possible path.
 * **Parametric Controlled Rotations:** Unlike standard CNOT-based circuits, this model utilizes CRX and CRY gates. These allow the model to learn not just whether to entangle, but the intensity of the entanglement, leading to high-precision decision boundaries.
 * **Reverse-Flow Correlation:** The second sub-layer reverses the entanglement direction ($i \to i-1$), facilitating a rapid diffusion of features across the entire register.
 * **Performance & Benchmarks:**
    In noise-free simulations, this architecture demonstrates exceptional learning capabilities:
      * **Accuracy:** consistently achieves **>90%** on binary classification tasks.
      * **Flexibility:** High parameter density allows for complex non-linear mapping of input data.
  
**Scientific Reference**

  The implementation of this ansatz is based on the architectural principles discussed in: [Quantum Machine Learning in Liquid – Havlíček et al. (2019) arXiv:1905.10876](https://https://arxiv.org/abs/1905.10876)
  
**The "Hardware Gap" (Motivation for Ansatz 2)**
While mathematically superior, Ansatz 1 poses significant challenges for physical quantum processors like the Odra system:

* **Gate Decomposition:** $CRX$ and $CRY$ are not native gates. On hardware, they are decomposed into multiple CNOTs and single-qubit rotations, multiplying the error rate for every single operation.

In [ ]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    """Weights when only the last macro-layer uses the q0-incident reverse trim."""
    n_macro = depth // 2
    if n_macro == 0:
        return 0
    full_layer = 4 * n_qubits
    last_layer = 3 * n_qubits + 2
    return (n_macro - 1) * full_layer + last_layer


def ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1

    assert param_idx == len(theta)
    return qc

### 4.2 Hybrid Variational Quantum Circuit

The `HybridModel` class wires the feature map + ansatz into a PyTorch module via `TorchConnector`, which is all we need to bind weights and expose the bound circuit to the fidelity helper in section 10.


In [ ]:
class HybridModel(nn.Module):
    """
    Hybrid Variational Quantum Circuit (VQC) for binary classification.

    The model combines:
    1. Angle encoding feature map (classical data → quantum state)
    2. Parametrized ansatz (trainable quantum circuit)
    3. Observable measurement (quantum state → classical expectation value)
    4. PyTorch integration via TorchConnector (enables backpropagation)

    Parameters
    ----------
    ansatz_circuit : QuantumCircuit
        Parametrized quantum circuit with trainable weights
    num_qubits : int
        Number of qubits (must match feature dimension)

    Attributes
    ----------
    qnn : EstimatorQNN
        Qiskit's EstimatorQNN that computes expectation values
    quantum_layer : TorchConnector
        PyTorch-compatible wrapper enabling gradient computation

    Notes
    -----
    - **Feature encoding**: RY(x_i) on qubit i encodes feature x_i
    - **Observable**: Pauli-Z on qubit 0, measuring spin in computational basis
    - **Output range**: [-1, +1] (expectation value of Z operator)
    - **Gradient method**: Reverse Estimator Gradient quantum gradients
    - **Simulator**: StatevectorEstimator (change for real quantum hardware)
    """

    def __init__(self, ansatz_circuit, num_qubits):
        super().__init__()
        # Create angle encoding feature map
        self.feature_map = self._create_angle_encoding(num_qubits)

        # Compose full quantum circuit: feature_map → ansatz
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        # Separate input parameters (from feature map) and weight parameters (from ansatz)
        # This distinction is crucial for EstimatorQNN to correctly handle data vs. trainable weights
        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)

        # Define observable: measure Z on qubit 0 (identity on other qubits)
        # Pauli string ordering: rightmost character = qubit 0
        # Example for 5 qubits: "IIIIZ" measures Z on q0, I on q1-q4
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])

        # Initialize statevector simulator for noiseless quantum simulation
        # NOTE: Replace with Sampler or real backend for quantum hardware deployment
        estimator = StatevectorEstimator()

        # Use parameter shift rule for computing quantum gradients
        # This is exact (not finite-difference) and works on hardware
        gradient = ParamShiftEstimatorGradient(estimator)

        # Create variational quantum circuit using EstimatorQNN
        # EstimatorQNN computes <ψ|O|ψ> where |ψ> = ansatz(weights)|feature_map(x)>
        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient
        )
        # Wrap the VQC as a PyTorch module
        # TorchConnector bridges Qiskit and PyTorch autograd systems,
        # allowing standard PyTorch optimizers (SGD, Adam, etc.) to train quantum parameters
        self.quantum_layer = TorchConnector(self.qnn)

    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        """
        Create angle encoding feature map: |0⟩ → RY(x₀) ⊗ RY(x₁) ⊗ ... ⊗ RY(xₙ) |0⟩

        Each classical feature x_i ∈ [0, π] is encoded as a rotation angle on qubit i.
        This maps the feature vector to the amplitude of the quantum state.

        Parameters
        ----------
        num_qubits : int
            Number of qubits (and features)

        Returns
        -------
        QuantumCircuit
            Feature map circuit with n_qubits input parameters
        """
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the hybrid variational quantum circuit.

        Parameters
        ----------
        x : torch.Tensor
            Input features, shape (batch_size, num_qubits)
         Returns
        -------
        torch.Tensor
            Expectation values, shape (batch_size, 1), range [-1, +1]
        """
        return self.quantum_layer(x)

## 5. Configuration and Data Loading


In [ ]:
NUM_QUBITS = 5
ANSATZ_DEPTH = 2

X_train, X_test, y_train_raw, y_test_raw, train_idx, test_idx = prepare_data(
    test_size=0.2,
    random_state=RANDOM_SEED,
)

# Map labels from {0, 1} to {-1, +1}
y_train = (2 * y_train_raw - 1).astype(np.float32)
y_test = (2 * y_test_raw - 1).astype(np.float32)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)


## 6. Load Trained Weights — Ansatz 1

We skip training entirely and load the pre-trained weights `2_weights_simulator_final_trimmed_reverse_q0.pth` so we can go straight to the fidelity comparison.


In [ ]:
# Load pre-trained weights for Ansatz 1 (Ring, simulator-optimized)
final_ansatz = ansatz(NUM_QUBITS, ANSATZ_DEPTH)
original_model = HybridModel(final_ansatz, NUM_QUBITS)
FILE_PATH = "2_weights_simulator_final_trimmed_reverse_q0.pth"
try:
    original_model.load_state_dict(torch.load(FILE_PATH))
    original_model.eval()
    print(f"Original-ansatz weights loaded from {FILE_PATH}.")
except FileNotFoundError:
    print(f"Error: weights file '{FILE_PATH}' was not found.")


## 7. Ansatz 2: Ring Topology (IQM Spark adapted)


**Overview:**
Ansatz 2 is a Hardware-Efficient Functional Form engineered to bypass the connectivity bottlenecks of the Odra quantum processor with main focuse on Native Gates usage.

Topology Mapping:
* **Ring Topology:** Entanglement is implemented via a circular chain where each qubit $i$ is coupled with qubit $(i+1) \pmod n$.


**Native Gate Optimization (CZ-Based)**
IQM Spark utilize Controlled-Z (CZ) as native entanglers.
* **Direct Execution:** By utilizing native CZ gates instead of synthetic CRX/CRY. This prevents the accumulation of coherent errors and reduces the overall gate-pulse duration.
* **Hybrid Expressibility:** To maintain the non-linear learning capabilities of controlled rotations, we implement a Pre-Entanglement Parameterization layer using $R_z$ and $R_y$ rotations. This effectively "tunes" the entanglement interaction locally before the native $CZ$ operation.

**Layered Parametric Strategy**
The ansatz utilizes a 2-step iterative block designed to probe the Hilbert space symmetrically:
* **Sub-Layer A:** Pairs independent $R_y$ rotations with $R_z$-tuned $CZ$ gates. This focuses on creating phase-sensitive correlations between neighboring qubits.
* **Sub-Layer B:** Pairs independent $R_x$ rotations with $R_y$-tuned $CZ$ gates. This simulates the effect of a $CRY$ interaction, allowing for complex amplitude redistribution while remaining hardware-native.

**Technical Specifications Gate Set:** $\{R_x, R_y, R_z, CZ\}$.

**Scientific Reference**
 **The power of quantum neural networks** – *Abbas et al. (2021)* [Nature Communications 12, 1476](https://www.nature.com/articles/s41467-021-21728-w)
    

In [ ]:
import sys
from pathlib import Path

def _project_root() -> Path:
    root = Path.cwd().resolve()
    while not (root / "star.py").exists() and root != root.parent:
        root = root.parent
    return root

sys.path.insert(0, str(_project_root()))
from star import star_ansatz as ansatz_Odra


## 8. Load Trained Weights — Ansatz 2

We load the pre-trained weights `2_weights_odra_final_trimmed_reverse_q0.pth` for the IQM-Spark-adapted ansatz so we can run section 10 immediately.


In [ ]:
# Load pre-trained weights for Ansatz 2 (Odra / IQM-Spark adapted)
final_ansatz = ansatz_Odra(NUM_QUBITS, ANSATZ_DEPTH)
odra_model = HybridModel(final_ansatz, NUM_QUBITS)
FILE_PATH = "2_weights_odra_final_trimmed_reverse_q0.pth"
try:
    odra_model.load_state_dict(torch.load(FILE_PATH))
    odra_model.eval()
    print(f"Odra-ansatz weights loaded from {FILE_PATH}.")
except FileNotFoundError:
    print(f"Error: weights file '{FILE_PATH}' was not found.")


## 9. IQM Spark Connection

Only the live `iqm_backend` handle is needed below — the noise model in section 10 is built directly from `iqm_backend.client` (live calibration set) without any custom hardware estimator.


In [ ]:
# Connect to the live IQM Spark backend so that section 10 below can
# pull the *current* calibration set (T1/T2, gate fidelities, durations,
# readout errors) and build the Aer noise model from it.
try:
    provider = IQMProvider("https://odra5.e-science.pl/", token=input("Enter IQM Token: "))
    iqm_backend = provider.get_backend()
    print(f"Connected to backend: {iqm_backend.name}")
except Exception as e:
    print(f"Connection error: {e}")
    raise


## 10. Methodology — End-to-End Fidelity Pipeline

This section is a self-contained methodology for the fidelity comparison performed below. The goal is that any reader can re-execute the cells, understand exactly what every number means, and audit every design choice — because the headline numbers (mean fidelity per ansatz) are striking and need to be reproducible.

---

### 10.1. The question we are answering

We have two trained variational quantum classifiers that differ **only in their ansatz**:

| Name | Ansatz | Native to IQM Spark? |
| --- | --- | --- |
| `original_model` | Ring topology with `CRX` / `CRY` controlled rotations (sections 4–6) | **No** — `CRX` / `CRY` decompose into multiple `r` + `cz` gates, and the ring uses wrap-around qubit pairs that are NOT in IQM Spark's coupling map. |
| `odra_model` | Ring topology built directly out of `R_x` / `R_y` / `R_z` + `CZ` (sections 7–8) | **Yes** — every primitive is a native IQM Spark gate. |

Both have already been trained classically to convergence (noiseless `StatevectorEstimator`) and the optimal weights $\theta^\star$ are loaded from `.pth` files in sections 6 and 8. **No further training happens here.**

For every test input $x_i$, we ask:

> If we were to run this exact bound circuit on IQM Spark *right now*, given the chip's current calibration, how close would the resulting quantum state be to the ideal state the optimiser thinks it is preparing?

The single number that answers this is the **state fidelity**

$$\boxed{\;\mathcal{F}(x_i)\;=\;\langle\psi_{\text{ideal}}(x_i,\theta^\star)\,|\,\rho_{\text{noisy}}(x_i,\theta^\star)\,|\,\psi_{\text{ideal}}(x_i,\theta^\star)\rangle\;}$$

with

- $\mathcal{F}\in[0,1]$;
- $\mathcal{F}=1$ → noisy execution is identical to noiseless;
- $\mathcal{F}\approx 1/2^n$ → noisy execution is essentially random (here $n=5$, so $\mathcal{F}\approx 0.031$ would be "white noise").

$\mathcal{F}$ is the **largest possible** classifier-agnostic figure of merit: it does not care about labels, accuracy, or which observable we eventually measure. It is purely "how badly does the chip mangle the state this architecture is supposed to prepare". Two architectures with the same accuracy but different $\mathcal{F}$ tell you that one is doing the same job with much less robustness margin.

---

### 10.2. The circuit being evaluated

For each ansatz, the bound circuit we evaluate is exactly the one that would be uploaded to the QPU:

1. **Angle-encoding feature map** — $\bigotimes_{i=0}^{n-1} R_y(x_i)$ on $n=5$ qubits. The features have been scaled to $[-\pi/4,\,\pi/4]$ in `prepare_data` (section 3).
2. **Ansatz** — `original_model` or `odra_model` with depth-2 (i.e. 2 macro-layers).
3. **No measurement** is appended at fidelity time — we want $\rho_{\text{noisy}}$, not shot statistics. (Measurement noise is still folded in via the `ReadoutError` channels described below; see "Why we omit explicit measurements" further down.)

Bound parameters:

- `input_params` ← $x_i$ (test sample $i$ from the scaled banknote test set);
- `weight_params` ← $\theta^\star$ (loaded weights, identical between the noiseless and noisy branches).

The two branches differ **only in the noise channel that is applied between gates** — nothing else.

---

### 10.3. Where the noise numbers come from (live IQM Spark calibration)

The Aer noise model is **never** read from a local file and **never** falls back to a synthetic snapshot. Every $T_1$ / $T_2$ / gate error / readout error is fetched at the moment the cell runs from the IQM server that `iqm_backend` is authenticated against.

**Why we don't use `NoiseModel.from_backend(iqm_backend)`.** `qiskit-iqm` does not populate `iqm_backend.target.qubit_properties` for IQM Spark, so `NoiseModel.from_backend(...)` returns an empty `NoiseModel` and prints `UserWarning: ... has no QubitProperties`. We therefore go to IQM's REST API directly:

| What we call | What we get |
| --- | --- |
| `iqm_backend.client.get_dynamic_quantum_architecture()` | The live `DynamicQuantumArchitecture`: which qubits exist, which gates are available on them, which gate implementations / loci exist in the calibration set. |
| `iqm_backend.client.get_calibration_set()` + `get_quality_metric_set()` | All observations from the just-downloaded calibration set, wrapped into an `iqm.iqm_client.ObservationFinder` with `skip_unparseable=True` so the cell tolerates new observation names the installed `iqm-client` release does not yet know how to parse. |
| `ObservationFinder.get_coherence_times([qubits])` | Per-qubit $T_1$, $T_2$. |
| `ObservationFinder.get_gate_fidelity(gate, impl, locus)` | Per-locus fidelity $\mathcal{F}_g$. |
| `ObservationFinder.get_gate_duration(gate, impl, locus)` | Per-locus gate duration $\tau_g$. |
| `ObservationFinder.get_measure_errors(gate, impl, locus)` | Per-qubit readout errors $P(1\|0)$ and $P(0\|1)$. |

If `iqm_backend` is `None`, or the resulting noise model has zero channels, the cell **raises immediately** — no silent fallback to fake data.

---

### 10.4. Building the Aer `NoiseModel`

For every IQM gate that (a) appears in the live calibration, and (b) has a known mapping to a Qiskit gate that the bound QNN circuit will actually emit, we attach the following channels:

| IQM gate | Qiskit gate | Channels attached |
| --- | --- | --- |
| `prx` (= phased RX) | `r` | depolarising error $p=1-\mathcal{F}_{\text{prx}}(q)$ composed with thermal relaxation $\mathcal{T}_{T_1(q),\,T_2(q),\,\tau_{\text{prx}}(q)}$. |
| `cz` | `cz` | 2-qubit depolarising error $p=1-\mathcal{F}_{\text{cz}}(q_a,q_b)$ composed with thermal relaxation on each of $q_a,\,q_b$ using their live $T_1,\,T_2$ and the live `cz` duration. |
| `measure` | `measure` | `ReadoutError(matrix=[[1-P(1\|0), P(1\|0)], [P(0\|1), 1-P(0\|1)]])` per qubit. |

IQM qubit names (`QB1`, `QB2`, …) are converted to Qiskit qubit indices via `iqm_backend.qubit_name_to_index(...)` so every per-locus channel is attached to the correct physical qubit. Ancillary IQM operations (`prx_12`, `cc_prx`, `reset_wait`, `measure_fidelity`, …) appear in the calibration set but never in the QNN circuit, so they are intentionally skipped to avoid harmless `UserWarning: ... will not apply` clutter.

**Why we also add an "all-qubit" fallback channel.** For every Qiskit gate that appears in the live calibration we additionally register an `add_all_qubit_quantum_error` whose magnitude is the **average** live IQM Spark error and the **average** live duration for that gate, combined with the **average** live $T_1$ / $T_2$ of the device.

This is the single most important design choice in the methodology, so it deserves its own paragraph:

> The Original (simulator-optimised) ansatz emits `cz` gates on qubit pairs that are NOT in IQM Spark's coupling map — most notably the wrap-around `cz` in the ring topology. If we relied only on the per-locus channels above, those off-coupling `cz` gates would silently incur **zero** noise from Aer (Aer attaches a per-locus channel only if that exact locus is in the noise model). The Original ansatz would then look unfairly clean, because the most expensive gates it emits would be free. The fallback channel ensures that **every gate the transpiler emits gets a representative IQM-Spark-magnitude error**, so the comparison between the two ansatze is apples-to-apples.

The end-of-cell printout in section 10.2 explicitly lists the fallback magnitudes, so the reader can verify exactly how much noise the off-coupling gates pay.

---

### 10.5. Per-sample fidelity protocol (`aer_fidelity`)

For one fixed input $x_i$ and the loaded weights $\theta^\star$, the helper executes the following four steps:

**Step 1 — Bind both data and trainable parameters.** `bind_circuit` substitutes $x_i$ into the feature-map parameters and $\theta^\star$ into the ansatz parameters, producing a fully numeric circuit `qc_bound`.

**Step 2 — Ideal state.** `psi_ideal = Statevector.from_instruction(qc_bound)` gives the noiseless pure state $|\psi_{\text{ideal}}\rangle$ as a state vector — no transpilation, no shots, no noise, no rounding.

**Step 3 — Native transpilation (no routing, no optimisation).**

```python
transpiled = transpile(
    qc_bound,
    basis_gates=list(noise_model.basis_gates),   # {'r', 'cz', 'measure'}
    optimization_level=0,
)
```

Two deliberate decisions here:

- `basis_gates` is the IQM Spark native set, taken from the live noise model we just built. This means that after transpilation, every gate in the circuit is one of `{r, cz}` — exactly the gates that have realistic noise channels attached.
- `optimization_level=0` — **no** gate cancellation, **no** commutation tricks, **no** routing onto a coupling map. We want to measure *the architecture's intrinsic cost on the IQM Spark gate set*, not the transpiler's ability to recover. This is the strictest interpretation of "what would I pay per gate?" and it is also the most conservative — the Original ansatz could in principle be helped by `optimization_level=3`, but we are explicitly not letting that happen here.

The transpiled circuit is then copied and `save_density_matrix` is appended so Aer returns the final state.

**Step 4 — Noisy execution.**

```python
sim = AerSimulator(noise_model=noise_model)
result = sim.run(transpiled_for_sim).result()
rho_noisy = DensityMatrix(result.data(0)['density_matrix'])
fidelity = state_fidelity(psi_ideal, rho_noisy)
```

This is a **full density-matrix simulation** of the transpiled circuit through the live IQM Spark noise model. There are no shots: $\rho_{\text{noisy}}$ is the exact mixed state that the noise channels produce. `state_fidelity(psi_ideal, rho_noisy)` then evaluates $\langle\psi_{\text{ideal}}|\rho_{\text{noisy}}|\psi_{\text{ideal}}\rangle$ analytically, and the helper additionally reports the post-transpilation depth, the total gate count, and the number of two-qubit gates so the reader can correlate fidelity with structural cost.

**Why density-matrix and not shot-based sampling?** Density-matrix simulation gives the *analytical* fidelity. Shot-based sampling would add shot noise on top of the hardware noise we are trying to measure, and the fidelity it estimates would converge to the same number we already get exactly. The whole point is to isolate the chip's contribution.

**Why we omit explicit measurements.** $|\psi_{\text{ideal}}\rangle$ is the state *immediately before* measurement, and $\rho_{\text{noisy}}$ is the same. Comparing them at this point gives the architecture's intrinsic state-preparation fidelity, which is independent of the choice of observable. Measurement readout error is still reflected in the noise model (the `ReadoutError` channels are registered above), but does not act on $\rho_{\text{noisy}}$ unless an explicit measurement instruction is present — and we deliberately exclude that here so that the metric isolates the **coherent + relaxation** error of the ansatz itself.

---

### 10.6. Sweeping across the test set (`sweep_fidelity`)

We do not report a single $\mathcal{F}$ value — we sweep $\mathcal{F}(x_i)$ for the first $N=50$ samples of the test set. This is important because **fidelity depends on the input**:

- The feature map $R_y(x_i)$ rotates each qubit to a different point on the Bloch sphere before the ansatz runs.
- $T_1$, $T_2$ and depolarising channels act differently on different parts of the Bloch sphere (e.g. $T_2$ dephasing is invisible to $|0\rangle$ and $|1\rangle$ but maximal on equatorial states).
- The ansatz then routes the state through a sequence of `cz` and rotation gates whose effective noise depends on the input state.

So a single fidelity number would hide a real distribution. The sweep reports the **mean, std, min and max** plus a histogram, so the reader can see whether the two ansatze sit on visually separate distributions or whether the gap between their means is within one standard deviation.

The same $N$ samples are used for both ansatze (same `X_test_np[:N]`), so the comparison is paired — every sample contributes one $\mathcal{F}$ value per ansatz against the **same** live calibration.

---

### 10.7. Reporting (`print_fidelity_report`)

For each ansatz we report:

- **Mean fidelity** $\bar{\mathcal{F}}$ across the $N$ samples — the headline number;
- **std** — how much $\mathcal{F}$ varies across inputs (small std → the architecture has a consistent degradation regardless of input);
- **min / max** — worst-case and best-case among the $N$ samples;
- **post-transpilation depth on the IQM Spark native basis** — structural cost (how many time steps a real IQM Spark run would need);
- **total gate count** — total number of native gates emitted, excluding `barrier` / `measure` / `save_density_matrix`;
- **two-qubit gate count** — the dominant error source on IQM Spark (`cz` fidelity is roughly an order of magnitude worse than `prx` fidelity).

The histogram overlays the two per-sample distributions so the reader can immediately see whether the gap between the means is meaningful (the histograms are far apart) or marginal (the histograms overlap).

---

### 10.8. Why this protocol is fair to *both* ansatze (limitations)

Every methodological choice above was chosen to be **conservative** — i.e. to either help the Original (simulator-optimised) ansatz or be neutral. None of them are stacked against it.

| Choice | Effect on the Original ansatz | Effect on the Odra ansatz |
| --- | --- | --- |
| `optimization_level=0` | Conservative — `optimization_level=3` could in principle cancel some of the gates that `CRX` decomposes into, which would help the Original ansatz; we explicitly forbid this so neither ansatz is rescued by the transpiler. | Neutral — the Odra ansatz is already in the native gate set, so the transpiler has nothing to cancel. |
| No routing onto the coupling map (`coupling_map=None`) | **Helps the Original ansatz** — the ring topology has wrap-around qubit pairs that IQM Spark does not natively couple, so a real run would have to pay SWAP overhead. We do not charge that cost here. | Neutral — the Odra ansatz already uses physical pairs that are in the coupling map. |
| All-qubit fallback noise channel | Charges the off-coupling `cz` gates with a representative (not infinite) error magnitude. This is *less* punishment than the Original ansatz would actually receive on hardware (where SWAPs would multiply the gate count). | Has essentially no effect — every `cz` in the Odra ansatz is already a coupling-map pair and would be matched by a per-locus channel anyway. |
| Density-matrix simulation (no shot noise) | Both ansatze get the same noise-free fidelity estimator. | Same. |
| Per-sample sweep ($N=50$) | Both ansatze get the same 50 inputs from the test set in the same order. | Same. |

In short: if anything, the protocol is *generous* to the Original ansatz. A real IQM Spark run with full routing would only widen the gap.

---

### 10.9. Reproducing the headline numbers

1. **Authenticate** to the live IQM Spark backend in section 9 (`iqm_backend`).
2. **Load** the trained weights in sections 6 (`original_model`) and 8 (`odra_model`) — same `.pth` files as in this notebook.
3. **Re-run section 10** top-to-bottom. The `NoiseModel` is rebuilt from whatever calibration `iqm_backend` is currently pointing at, so the absolute numbers may shift with the chip's recalibration cycle; the **relative** numbers (which ansatz has the higher mean fidelity, by how much) should remain stable as long as `cz` fidelity on IQM Spark stays roughly an order of magnitude worse than `prx` fidelity, which is the regime IQM Spark operates in today.
4. To audit the noise magnitudes that section 10.2 ended up using, re-read the printout of that cell — it lists per-locus fidelities, $T_1$, $T_2$, and the per-gate fallback magnitudes that were actually plugged into the channels.

---

### 10.10. What the result will tell you

- $\bar{\mathcal{F}} \to 1$ — the architecture barely feels the IQM Spark error rates; the ansatz is shallow and well-matched to the native gate set.
- $\bar{\mathcal{F}}$ noticeably below $1$ — every shot on IQM Spark would be a non-trivial mixture of the ideal state and an error state. In practice, the expectation values that the classifier consumes get rescaled by roughly $\bar{\mathcal{F}}$, which limits classification confidence and degrades $\langle Z_0\rangle$.
- $\bar{\mathcal{F}}$ approaching $1/2^n = 1/32$ — the noisy state is essentially indistinguishable from a maximally mixed state, and any classification signal at the end of the circuit is destroyed.
- A large gap in `depth` or `#2q` but a small gap in $\bar{\mathcal{F}}$ would mean the IQM native decomposition equalises the two ansatze. A small gap in structural metrics but a large gap in $\bar{\mathcal{F}}$ would mean one ansatz hits error-prone gates more often per layer.

### 10.2. Live IQM Spark calibration → Aer NoiseModel

`qiskit-iqm` does **not** populate `iqm_backend.target.qubit_properties` for IQM Spark, so the usual `NoiseModel.from_backend(...)` path returns an empty noise model (and prints a `UserWarning: ... has no QubitProperties`). We therefore go straight to IQM's own REST API on the live `iqm_backend.client`:

- `iqm_backend.client.get_dynamic_quantum_architecture()` returns the current `DynamicQuantumArchitecture` (the set of qubits, the gates available on them, and the gate implementations / loci that exist in the calibration set).
- `iqm_backend.client.get_calibration_quality_metrics()` returns an `ObservationFinder` over the **just-downloaded** quality metric set, with helpers `get_coherence_times`, `get_gate_fidelity`, `get_gate_duration`, and `get_measure_errors` keyed by IQM qubit names (`QB1`, `QB2`, …) and locus tuples.

The cell below uses those calls to build an Aer `NoiseModel` in real time:

- **Single-qubit `prx` gates** (exposed in Qiskit as the `r` gate) get a depolarizing channel with $p = 1 - \mathcal{F}_{\text{prx}}$ for the current locus, composed with a thermal-relaxation channel parameterized by the live $T_1$, $T_2$ and the live gate duration.
- **Two-qubit `cz` gates** get a 2-qubit depolarizing channel with $p = 1 - \mathcal{F}_{\text{cz}}$, composed with a thermal-relaxation channel on each of the two qubits using their live $T_1$ / $T_2$ and the live `cz` duration.
- **Measurements** get a readout-error channel built from the live $P(1|0)$ and $P(0|1)$ per qubit.

IQM qubit names are mapped to Qiskit qubit indices via `iqm_backend.qubit_name_to_index(...)`, so the per-locus channels above are attached on the right qiskit qubits.

In addition, for **every** Qiskit gate type that appears in the live calibration we also register an `add_all_qubit_quantum_error` *fallback* whose magnitude is the **average** live IQM Spark error and duration for that gate, combined with the average live $T_1$ / $T_2$ of the device. This fallback is necessary because the simulator-optimized ansatz emits `cz` gates on pairs that are **not** in IQM Spark's coupling map (e.g. the wrap-around `cz` in the ring topology). Without the fallback, those off-coupling gates would silently pick up zero noise from Aer, and the simulator-optimized ansatz would look unfairly clean compared to the IQM-Spark-adapted one. With the fallback, every gate the transpiler emits gets a representative IQM-Spark-magnitude error, so the fidelity comparison between the two ansatze is apples-to-apples.

If the live IQM Spark backend is not connected or the live calibration yields zero noise channels, the cell raises immediately. There is **no fallback to any pre-built / fake noise model** — every number printed below this point is computed from the **current** IQM Spark calibration set.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    thermal_relaxation_error,
    ReadoutError,
)
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity

from iqm.iqm_client import ObservationFinder


if "iqm_backend" not in globals() or iqm_backend is None:
    raise RuntimeError(
        "IQM Spark backend is not connected. Re-run section 7 (Hardware Setup) "
        "so that `iqm_backend` is authenticated against the live IQM Spark (Odra) "
        "device. This section uses ONLY the live calibration set downloaded in "
        "real time from IQM -- no pre-built / fake noise model is allowed."
    )


IQM_TO_QISKIT_GATE = {
    "prx": "r",
    "phased_rx": "r",
    "cz": "cz",
    "measure": "measure",
    "measurement": "measure",
}


def _fetch_live_observation_finder(backend) -> ObservationFinder:
    """
    Replicate ``IQMClient._get_calibration_quality_metrics`` but build the
    ``ObservationFinder`` with ``skip_unparseable=True`` so we tolerate any
    forward-compatible observation names the IQM server emits that the
    installed iqm-client release doesn't yet know how to parse.

    Everything pulled here is live from the IQM server -- no offline data.
    """
    client = backend.client
    calibration_set = client.get_calibration_set()
    quality_metrics = client.get_quality_metric_set()
    return ObservationFinder(
        list(calibration_set.observations) + list(quality_metrics.observations),
        skip_unparseable=True,
    )


def _build_iqm_noise_model_from_live_calibration(backend) -> tuple[NoiseModel, dict]:
    """
    Build a Qiskit Aer ``NoiseModel`` from the LIVE IQM Spark calibration set.

    The data is fetched at call time from the IQM control server via
    ``backend.client.get_dynamic_quantum_architecture()`` and
    ``backend.client.get_calibration_quality_metrics()``. Nothing is read
    from disk and no fake / offline backend is consulted.

    Returns
    -------
    (NoiseModel, summary_dict)
        ``summary_dict`` contains the raw live calibration that was used
        to build the channels, so the user can audit / print it.
    """
    client = backend.client
    dqa = client.get_dynamic_quantum_architecture()
    quality_obs = _fetch_live_observation_finder(backend)

    iqm_qubits = list(dqa.qubits)

    t1_dict, t2_dict = quality_obs.get_coherence_times(iqm_qubits)

    # We only build noise channels for IQM gates that have a known mapping
    # to a Qiskit gate that our QNN circuit will actually emit
    # (`prx` -> `r`, `cz` -> `cz`, `measure` -> `measure`). The IQM server
    # also reports calibration data for ancillary operations like `prx_12`
    # (a |1>->|2> drive used for error mitigation), `reset_wait`,
    # `cc_prx` (classically-controlled prx) and `measure_fidelity` --
    # these never appear in the QNN circuit, so attaching noise to them
    # would just produce harmless "all-qubit error will not apply..."
    # warnings without affecting the fidelity comparison.
    iqm_gates_used = {g for g in dqa.gates.keys() if g in IQM_TO_QISKIT_GATE}
    qiskit_basis = sorted({IQM_TO_QISKIT_GATE[g] for g in iqm_gates_used})
    noise_model = NoiseModel(basis_gates=qiskit_basis)

    gate_summary: dict[tuple, dict] = {}
    measure_summary: dict[tuple, dict] = {}

    # We also collect per-qiskit-gate averages so we can register an
    # ``all_qubit`` fallback noise channel.  This way, if the ansatz uses a
    # 2-qubit gate on a pair that is NOT in IQM Spark's coupling map (e.g.
    # the wrap-around CZ in the ring ansatz), Aer still applies a
    # representative IQM-Spark-magnitude error to that gate -- otherwise
    # those gates would silently incur no noise and the simulator-optimised
    # ansatz would look unfairly clean.
    per_qiskit_gate_stats: dict[str, dict] = {}

    for iqm_gate_name, gate_info in dqa.gates.items():
        if iqm_gate_name not in IQM_TO_QISKIT_GATE:
            continue
        qiskit_gate_name = IQM_TO_QISKIT_GATE[iqm_gate_name]

        for impl_name, impl_info in gate_info.implementations.items():
            for locus in impl_info.loci:
                if iqm_gate_name in {"measure", "measurement"}:
                    err_pair = quality_obs.get_measure_errors(iqm_gate_name, impl_name, locus)
                    if err_pair is None:
                        continue
                    e_01, e_10 = err_pair
                    e_01 = min(max(float(e_01), 0.0), 1.0)
                    e_10 = min(max(float(e_10), 0.0), 1.0)
                    try:
                        qubit_indices = [backend.qubit_name_to_index(q) for q in locus]
                    except Exception:
                        continue
                    if any(q is None for q in qubit_indices):
                        continue
                    ro_matrix = [[1.0 - e_01, e_01], [e_10, 1.0 - e_10]]
                    noise_model.add_readout_error(ReadoutError(ro_matrix), qubit_indices)
                    measure_summary[(iqm_gate_name, impl_name, tuple(locus))] = {
                        "error_0_to_1": e_01,
                        "error_1_to_0": e_10,
                    }
                    continue

                fidelity = quality_obs.get_gate_fidelity(iqm_gate_name, impl_name, locus)
                duration = quality_obs.get_gate_duration(iqm_gate_name, impl_name, locus)
                if fidelity is None and duration is None:
                    continue

                try:
                    qubit_indices = [backend.qubit_name_to_index(q) for q in locus]
                except Exception:
                    continue
                if any(q is None for q in qubit_indices):
                    continue

                n_gate_qubits = len(locus)
                channel = None

                if fidelity is not None:
                    err_prob = float(max(0.0, 1.0 - float(fidelity)))
                    err_prob = min(err_prob, 1.0 - 1e-12)
                    if err_prob > 0.0:
                        channel = depolarizing_error(err_prob, n_gate_qubits)

                if duration is not None and duration > 0.0:
                    relax = None
                    for q in locus:
                        t1 = t1_dict.get(q)
                        t2 = t2_dict.get(q)
                        if t1 is None or t2 is None:
                            relax = None
                            break
                        t2_eff = min(float(t2), 2.0 * float(t1))
                        qre = thermal_relaxation_error(float(t1), t2_eff, float(duration))
                        relax = qre if relax is None else relax.tensor(qre)
                    if relax is not None:
                        channel = relax if channel is None else channel.compose(relax)

                if channel is not None:
                    noise_model.add_quantum_error(channel, qiskit_gate_name, qubit_indices)
                    gate_summary[(iqm_gate_name, impl_name, tuple(locus))] = {
                        "qiskit_gate": qiskit_gate_name,
                        "qubit_indices": tuple(qubit_indices),
                        "fidelity": fidelity,
                        "duration_s": duration,
                    }
                    stats = per_qiskit_gate_stats.setdefault(
                        qiskit_gate_name,
                        {"n_qubits": n_gate_qubits, "errors": [], "durations": []},
                    )
                    if fidelity is not None:
                        stats["errors"].append(float(max(0.0, 1.0 - float(fidelity))))
                    if duration is not None and duration > 0.0:
                        stats["durations"].append(float(duration))

    # Build an ``all_qubit`` fallback per qiskit gate using the average error
    # and average duration across the live calibration set, combined with the
    # average T1 / T2 across the device.
    avg_t1 = (
        sum(t1_dict.values()) / len(t1_dict) if t1_dict else None
    )
    avg_t2 = (
        sum(t2_dict.values()) / len(t2_dict) if t2_dict else None
    )

    default_channel_summary: dict[str, dict] = {}
    for qiskit_gate_name, stats in per_qiskit_gate_stats.items():
        errors = stats["errors"]
        durations = stats["durations"]
        n_q = stats["n_qubits"]

        if not errors and not durations:
            continue

        channel = None

        if errors:
            avg_err = sum(errors) / len(errors)
            avg_err = min(max(avg_err, 0.0), 1.0 - 1e-12)
            if avg_err > 0.0:
                channel = depolarizing_error(avg_err, n_q)

        if durations and avg_t1 is not None and avg_t2 is not None:
            avg_dur = sum(durations) / len(durations)
            t2_eff = min(avg_t2, 2.0 * avg_t1)
            relax = thermal_relaxation_error(avg_t1, t2_eff, avg_dur)
            for _ in range(n_q - 1):
                relax = relax.tensor(thermal_relaxation_error(avg_t1, t2_eff, avg_dur))
            channel = relax if channel is None else channel.compose(relax)

        if channel is not None:
            noise_model.add_all_qubit_quantum_error(channel, qiskit_gate_name)
            default_channel_summary[qiskit_gate_name] = {
                "avg_error": (sum(errors) / len(errors)) if errors else None,
                "avg_duration_s": (sum(durations) / len(durations)) if durations else None,
                "n_qubits": n_q,
            }

    summary = {
        "calibration_set_id": str(dqa.calibration_set_id),
        "iqm_qubits": iqm_qubits,
        "t1": t1_dict,
        "t2": t2_dict,
        "avg_t1_s": avg_t1,
        "avg_t2_s": avg_t2,
        "gates": gate_summary,
        "measure": measure_summary,
        "qiskit_basis": qiskit_basis,
        "default_channels": default_channel_summary,
    }
    return noise_model, summary


print(f"Fetching live calibration from {iqm_backend.name} ...")
iqm_noise_model, iqm_calibration_summary = _build_iqm_noise_model_from_live_calibration(iqm_backend)
iqm_noise_source = (
    f"live IQM Spark calibration set {iqm_calibration_summary['calibration_set_id']}"
    f" on {iqm_backend.name}"
)

if not iqm_noise_model.noise_instructions:
    raise RuntimeError(
        "Live IQM Spark calibration did not yield any noise channels. Refusing to "
        "fall back to a pre-built / fake noise model -- re-authenticate the IQM "
        "backend in section 7 and rerun this cell."
    )

iqm_native_basis = iqm_calibration_summary["qiskit_basis"]

print(f"Noise source            : {iqm_noise_source}")
print(f"IQM backend             : {iqm_backend.name}")
print(f"Backend qubit count     : {iqm_backend.num_qubits}")
print(f"Native gate set         : {sorted(iqm_native_basis)}")
print(f"Noise-model basis gates : {sorted(iqm_noise_model.basis_gates)}")
print(f"Noisy instructions      : {sorted(iqm_noise_model.noise_instructions)}")

t1_dict = iqm_calibration_summary["t1"]
t2_dict = iqm_calibration_summary["t2"]
if t1_dict or t2_dict:
    print("\nLive IQM Spark per-qubit coherence times (current set):")
    print(f"{'IQM qb':>8} {'qiskit idx':>11} {'T1 [us]':>10} {'T2 [us]':>10}")
    for q in iqm_calibration_summary["iqm_qubits"]:
        try:
            idx = iqm_backend.qubit_name_to_index(q)
        except Exception:
            idx = None
        t1_us = (t1_dict[q] * 1e6) if q in t1_dict else float("nan")
        t2_us = (t2_dict[q] * 1e6) if q in t2_dict else float("nan")
        idx_str = f"{idx:>11d}" if idx is not None else f"{'-':>11}"
        print(f"{q:>8} {idx_str} {t1_us:>10.2f} {t2_us:>10.2f}")

gate_summary = iqm_calibration_summary["gates"]
if gate_summary:
    print("\nLive IQM Spark per-gate error rates (1 - fidelity, current set):")
    by_gate: dict[str, list] = {}
    for (iqm_name, impl, locus), info in gate_summary.items():
        if info["fidelity"] is None:
            continue
        err = 1.0 - float(info["fidelity"])
        by_gate.setdefault(iqm_name, []).append((locus, err, info["duration_s"]))
    for iqm_name, rows in by_gate.items():
        errs = [r[1] for r in rows]
        avg = sum(errs) / len(errs)
        worst = max(rows, key=lambda r: r[1])
        qiskit_name = IQM_TO_QISKIT_GATE.get(iqm_name, iqm_name)
        print(
            f"  {iqm_name:<6} (qiskit '{qiskit_name}')  n={len(rows):<3} "
            f"avg err={avg:.4e}  worst={worst[1]:.4e} @ {worst[0]}"
        )

measure_summary = iqm_calibration_summary["measure"]
if measure_summary:
    print("\nLive IQM Spark readout errors (current set):")
    for (_, _, locus), info in measure_summary.items():
        print(
            f"  qubits {locus}: P(1|0)={info['error_0_to_1']:.4e}  "
            f"P(0|1)={info['error_1_to_0']:.4e}"
        )

default_channels = iqm_calibration_summary["default_channels"]
if default_channels:
    print("\nIQM-Spark-magnitude all_qubit fallback channels (averages from live set):")
    for qiskit_gate_name, info in default_channels.items():
        err_part = (
            f"avg err={info['avg_error']:.4e}" if info["avg_error"] is not None else "no fidelity data"
        )
        dur_part = (
            f"avg dur={info['avg_duration_s']*1e9:.1f} ns"
            if info["avg_duration_s"] is not None
            else "no duration data"
        )
        print(f"  {qiskit_gate_name:<6} ({info['n_qubits']}q):  {err_part}  {dur_part}")

### 10.3. Fidelity Helper

The helper does the four steps below for one bound circuit (feature map + ansatz, with $x$ and $\theta^\star$ already substituted in):

1. Compute $|\psi_{\text{ideal}}\rangle$ via a noiseless `Statevector` simulation (no transpilation noise).
2. Transpile the same circuit to the **IQM Spark native gate set** (`r` / `cz`, taken from the live `iqm_backend.target`) at `optimization_level=0`, so we measure the architecture's intrinsic cost on IQM Spark, not Qiskit's optimiser cleverness.
3. Run a **density-matrix simulation** of the transpiled circuit through `AerSimulator(noise_model=iqm_noise_model)`, where `iqm_noise_model` was built one cell up from the **live, just-downloaded** IQM Spark calibration, and recover $\rho_{\text{noisy}}$.
4. Return $\mathcal{F} = \langle\psi_{\text{ideal}}|\rho_{\text{noisy}}|\psi_{\text{ideal}}\rangle$ together with the post-transpilation gate counts.

In [ ]:
def bind_circuit(qc, input_params, weight_params, x_value, weight_values):
    """Substitute concrete values for both data and trainable parameters."""
    binding = {p: float(v) for p, v in zip(input_params, x_value)}
    binding.update({p: float(v) for p, v in zip(weight_params, weight_values)})
    return qc.assign_parameters(binding)


def aer_fidelity(qc_bound,
                 noise_model,
                 basis_gates=None,
                 optimization_level=0):
    """
    Compute the state fidelity F = <psi_ideal | rho_noisy | psi_ideal>
    between an ideal noiseless execution of qc_bound and an Aer noisy
    execution that applies the LIVE IQM Spark hardware noise model on
    every gate.

    The circuit is decomposed into the IQM Spark native gate set (so the
    noise channels match the gates that would actually be executed on
    IQM Spark / Odra) but is NOT routed onto its coupling graph -- the
    metric isolates how the architecture's per-gate work is degraded by
    the live IQM Spark error rates, independent of any routing overhead.

    Returns
    -------
    dict with keys: fidelity, depth, total_gates, two_qubit_gates.
    """
    if basis_gates is None:
        basis_gates = list(noise_model.basis_gates)

    psi_ideal = Statevector.from_instruction(qc_bound)

    transpiled = transpile(
        qc_bound,
        basis_gates=basis_gates,
        optimization_level=optimization_level,
    )

    transpiled_for_sim = transpiled.copy()
    transpiled_for_sim.save_density_matrix()

    sim = AerSimulator(noise_model=noise_model)
    result = sim.run(transpiled_for_sim).result()
    rho_noisy = DensityMatrix(result.data(0)["density_matrix"])

    fidelity = float(state_fidelity(psi_ideal, rho_noisy))

    skip = {"barrier", "measure", "save_density_matrix"}
    total_gates = 0
    two_qubit_gates = 0
    for instr in transpiled.data:
        name = instr.operation.name
        if name in skip:
            continue
        total_gates += 1
        if instr.operation.num_qubits == 2:
            two_qubit_gates += 1

    return {
        "fidelity": fidelity,
        "depth": transpiled.depth(),
        "total_gates": total_gates,
        "two_qubit_gates": two_qubit_gates,
    }

### 10.4. Sweep Both Ansatze Across the Test Set

We evaluate $\mathcal{F}$ for every test sample $x_i$ using the corresponding trained weights $\theta^\star$. The fidelity depends on the input because the feature map rotates the qubits before the ansatz runs, and different rotation angles route different states through different parts of the noise channel.

In [ ]:
def sweep_fidelity(model, x_samples, noise_model, label, max_samples=50):
    """Run the Aer-noise fidelity protocol across (up to) ``max_samples`` test inputs."""
    n = min(max_samples, len(x_samples))
    weight_values = model.quantum_layer.weight.detach().numpy()
    input_params = list(model.feature_map.parameters)
    weight_params = [p for p in model.qc.parameters if p not in set(input_params)]

    fidelities = np.zeros(n)
    structural = None

    for i in range(n):
        bound = bind_circuit(
            model.qc,
            input_params,
            weight_params,
            x_value=x_samples[i],
            weight_values=weight_values,
        )
        out = aer_fidelity(bound, noise_model)
        fidelities[i] = out["fidelity"]
        if structural is None:
            structural = {
                "depth": out["depth"],
                "total_gates": out["total_gates"],
                "two_qubit_gates": out["two_qubit_gates"],
            }

    summary = {
        "label": label,
        "n_samples": n,
        "fidelity_mean": float(fidelities.mean()),
        "fidelity_std": float(fidelities.std(ddof=0)),
        "fidelity_min": float(fidelities.min()),
        "fidelity_max": float(fidelities.max()),
        **structural,
        "fidelities": fidelities,
    }
    return summary

X_test_np = X_test_tensor.numpy()
N_FIDELITY_SAMPLES = 50

print("Sweeping Original (Ring, simulator-optimized) ansatz under IQM Spark noise...")
fid_original = sweep_fidelity(
    model=original_model,
    x_samples=X_test_np,
    noise_model=iqm_noise_model,
    label="Original (Ring, sim-opt)",
    max_samples=N_FIDELITY_SAMPLES,
)
print(f"  mean F = {fid_original['fidelity_mean']:.4f}  "
      f"std = {fid_original['fidelity_std']:.4f}")

print("\nSweeping Odra (IQM-Spark adapted) ansatz under IQM Spark noise...")
fid_odra = sweep_fidelity(
    model=odra_model,
    x_samples=X_test_np,
    noise_model=iqm_noise_model,
    label="Odra (IQM-Spark adapted)",
    max_samples=N_FIDELITY_SAMPLES,
)
print(f"  mean F = {fid_odra['fidelity_mean']:.4f}  "
      f"std = {fid_odra['fidelity_std']:.4f}")

### 10.5. Comparison Report

For each ansatz we report:

- **Mean fidelity** $\bar{\mathcal{F}}$ across the test set — the headline number, computed against the **live IQM Spark calibration**.
- **Min / max fidelity** — variance across different inputs.
- **Post-transpilation depth & two-qubit gate count on the IQM Spark native basis** (`r` / `cz`) — structural cost on the actual hardware.

The histograms below show the per-sample distribution; if the two histograms overlap, the architectures are equally degraded by the current IQM Spark noise.

In [ ]:
def print_fidelity_report(*summaries):
    print("=" * 78)
    print(f"{'Ansatz':<32}{'mean F':>10}{'std':>10}{'min':>10}{'max':>10}")
    print("-" * 78)
    for s in summaries:
        print(
            f"{s['label']:<32}"
            f"{s['fidelity_mean']:>10.4f}"
            f"{s['fidelity_std']:>10.4f}"
            f"{s['fidelity_min']:>10.4f}"
            f"{s['fidelity_max']:>10.4f}"
        )
    print("-" * 78)
    print(f"{'Ansatz':<32}{'depth':>10}{'#gates':>10}{'#2q':>10}")
    for s in summaries:
        print(
            f"{s['label']:<32}"
            f"{s['depth']:>10d}"
            f"{s['total_gates']:>10d}"
            f"{s['two_qubit_gates']:>10d}"
        )
    print("=" * 78)


print_fidelity_report(fid_original, fid_odra)

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.hist(fid_original["fidelities"], bins=20, alpha=0.55, label=fid_original["label"])
ax.hist(fid_odra["fidelities"],     bins=20, alpha=0.55, label=fid_odra["label"])
ax.axvline(fid_original["fidelity_mean"], linestyle="--", linewidth=1)
ax.axvline(fid_odra["fidelity_mean"],     linestyle="--", linewidth=1)
ax.set_xlabel(r"State fidelity  $\mathcal{F} = \langle\psi_{\mathrm{ideal}}|\rho_{\mathrm{noisy}}|\psi_{\mathrm{ideal}}\rangle$")
ax.set_ylabel("count (test samples)")
ax.set_title(f"Per-sample fidelity under live {iqm_backend.name} (IQM Spark) noise")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 10.11. Optional next experiments

The methodology above is deliberately the **strictest, most conservative** interpretation of "what per-gate cost does each ansatz pay on IQM Spark?". If you want to stress-test the result, here are the natural follow-ups:

- **Let the transpiler help.** Re-run with `optimization_level=3` inside `aer_fidelity`. This is the upper bound on what each ansatz can recover from gate-level optimisation on the IQM native basis. If the gap between the two ansatze persists at `optimization_level=3`, the gap is architectural — the transpiler cannot save the simulator-optimised ansatz.
- **Charge the routing cost.** Pass `coupling_map=iqm_backend.coupling_map` (and a chosen `initial_layout`) into `aer_fidelity` to also pay the routing cost on the IQM Spark star topology. This requires reducing the simulator to the qubits actually used (e.g. via `AerSimulator.from_backend(iqm_backend)` together with a small `initial_layout`). The expectation is that the gap between the two ansatze **widens** because the simulator-optimised ansatz now also has to pay SWAP overhead, which is exactly the cost the present methodology hides.
- **Refresh the calibration.** Re-execute section 9 right before re-running section 10. The `NoiseModel` is rebuilt from whatever calibration `iqm_backend` is currently pointing at, so re-running 9 → 10 gives a like-for-like comparison against the freshest hardware metrics. The absolute fidelity numbers may drift between IQM Spark recalibrations; the relative ordering of the two ansatze should not.
- **Vary the test slice.** Increase `N_FIDELITY_SAMPLES` from 50 to the full test set. The std of $\mathcal{F}$ should not grow much — if it does, that itself is a finding (it means $\mathcal{F}$ is more input-dependent than we thought, and the per-sample histogram in 10.5 is the right place to look).